# Foundation models with `ov.llm`

`ov.llm.ModelFactory` is the single entry point for single-cell foundation models. `create_model(model_type, model_path, device, **kwargs)` returns a model object (`ScGPTModel`, `ScFoundationModel`, `GeneformerModel`, `CellPLMModel`, `UCEModel`, `ScMulanModel`, ...) whose methods (`get_embeddings`, `predict`, `fine_tune`, `integrate`, ...) do the work.

This notebook is an overview of those entry points. The first cells run without any checkpoint and keep their outputs. The cells from *Create a model* onward need model checkpoints (and a GPU) and are **not executed here**; their executed counterparts are the per-model tutorials (scGPT, scFoundation, Geneformer, CellPLM, UCE).

## Registered model types

In [1]:
import omicverse as ov

model_types = ov.llm.ModelFactory.available_models()
print(len(model_types), model_types)

22 ['scgpt', 'scfoundation', 'geneformer', 'cellplm', 'uce', 'scmulan', 'scbert', 'genecompass', 'nicheformer', 'tgpt', 'cellfm', 'sccello', 'scprint', 'aidocell', 'pulsar', 'atacformer', 'scplantllm', 'langcell', 'cell2sentence', 'genept', 'chatcell', 'tabula']


Six types are integrated; the other 16 are placeholders from `omicverse.llm.external_model_stubs`.

In [2]:
from omicverse.llm.external_model_stubs import ExternalModelStub

factory = ov.llm.ModelFactory
integrated = [k for k in model_types if not issubclass(factory._models[k], ExternalModelStub)]
placeholders = [k for k in model_types if k not in integrated]
print('integrated :', integrated)
print('placeholder:', placeholders)

integrated : ['scgpt', 'scfoundation', 'geneformer', 'cellplm', 'uce', 'scmulan']
placeholder: ['scbert', 'genecompass', 'nicheformer', 'tgpt', 'cellfm', 'sccello', 'scprint', 'aidocell', 'pulsar', 'atacformer', 'scplantllm', 'langcell', 'cell2sentence', 'genept', 'chatcell', 'tabula']


## A placeholder model

Placeholders can be created without a checkpoint, but every task raises `ModelExecutionBlockedError` with the reason and upstream URL. Nothing is computed for them.

In [3]:
import anndata

model = ov.llm.ModelFactory.create_model('scbert')
print(model)
try:
    model.get_embeddings(anndata.AnnData())   # raises before touching the data
except ov.llm.ModelExecutionBlockedError as err:
    print(err)
    print(err.checkpoint_url)

ScBERTModel(model_name=scbert, device=cpu, status=not loaded)
scbert cannot run task 'embed': scBERT is registered but the upstream inference runtime is not yet integrated into omicverse.llm.
https://github.com/TencentAILabHealthcare/scBERT


## Create a model (needs a checkpoint)

Every integrated model is created the same way. Checkpoint sources are listed on each model page and in `omicverse.llm.model_download`.

In [ ]:
model = ov.llm.ModelFactory.create_model(
    model_type="scgpt",
    model_path="llm_model/models/scgpt",
    device="cuda",
)
embeddings = model.get_embeddings(adata, skip_normalization=True)   # (n_cells, 512)

### Annotation and integration

`fine_tune` adapts the model on a labelled reference; `predict_celltypes` (scGPT, scFoundation, Geneformer, CellPLM) or `predict(task="annotation")` labels the query; `integrate` corrects batches on top of the embeddings.

In [ ]:
model.fine_tune(train_adata=reference_adata, epochs=5, batch_size=32, lr=1e-4,
                mask_ratio=0.0, data_is_raw=False, log1p=False, skip_normalization=True)
res = model.predict_celltypes(adata, skip_normalization=True)
adata.obs["predicted_celltype"] = res["predicted_celltypes"]
res = model.integrate(adata, batch_key="batch", correction_method="mnn",
                      log1p=False, data_is_raw=False, skip_normalization=True)
adata.obsm["X_scgpt_fine"] = res["embeddings"]

## Other integrated models

scFoundation and CellPLM take a checkpoint path; Geneformer is created first and loaded with its three dictionary files; UCE needs five asset paths; scMulan supports embedding and zero-shot annotation only.

In [ ]:
sf = ov.llm.ModelFactory.create_model("scfoundation", "llm_model/models/scfoundation/models.ckpt", "cuda")
emb = sf.get_embeddings(adata, pre_normalized="T", input_type="singlecell")   # (n_cells, 3072)
cp = ov.llm.ModelFactory.create_model("cellplm", "llm_model/models/cellplm", "cuda",
                                      pretrain_version="20231027_85M")
emb = cp.get_embeddings(adata)   # (n_cells, 512)

In [ ]:
gf = ov.llm.ModelFactory.create_model("geneformer", device="cuda", model_version="V2")
gf.load_model("llm_model/models/geneformer/Geneformer-V2-104M",
              gene_median_file="llm_model/models/geneformer/gene_median_dictionary_gc104M.pkl",
              token_dictionary_file="llm_model/models/geneformer/token_dictionary_gc104M.pkl",
              gene_mapping_file="llm_model/models/geneformer/ensembl_mapping_dict_gc104M.pkl")
emb = gf.get_embeddings(adata, max_ncells=100000)   # (n_cells, 768)

In [ ]:
uce = ov.llm.ModelFactory.create_model(
    "uce", "llm_model/models/UCE/4layer_model.torch", "cuda", species="human",
    token_file="llm_model/models/UCE/all_tokens.torch",
    protein_embeddings_dir="llm_model/models/UCE/protein_embeddings/",
    spec_chrom_csv_path="llm_model/models/UCE/species_chrom.csv",
    offset_pkl_path="llm_model/models/UCE/species_offsets.pkl")
emb = uce.get_embeddings(adata, filter_cells=False, filter_genes=False)   # (n_cells, 1280)

In [ ]:
mulan = ov.llm.ModelFactory.create_model("scmulan", "llm_model/models/scmulan/ckpt_scMulan.pt", "cuda")
emb = mulan.get_embeddings(adata)
res = mulan.predict(adata, task="annotation")   # res["predicted_celltypes"], res["embeddings"]

## Per-model helper functions

`omicverse.llm.model_factory` bundles create + run into single calls for scGPT, scFoundation, Geneformer and CellPLM (for example `get_embeddings_with_cellplm`, `end_to_end_scgpt_annotation`, `perturb_genes_with_geneformer`). UCE and scMulan have none. The full list with signatures is on each model page.

In [ ]:
from omicverse.llm.model_factory import get_embeddings_with_cellplm

emb = get_embeddings_with_cellplm(adata, model_path="llm_model/models/cellplm",
                                  pretrain_version="20231027_85M", device="cuda")

## Migrating from the removed manager class

`manager = Manager(model_type=T, model_path=P, device=D)` becomes `model = ov.llm.ModelFactory.create_model(T, P, D)`; `manager.model.<method>` becomes `model.<method>`; `manager.annotate_cells(adata)` becomes `model.predict(adata, task="annotation")`; `manager.get_embeddings`, `integrate`, `fine_tune`, `train_integration` and `save_model` keep their names on the model object.

Sixteen further model types are placeholders only; see the *Not yet integrated* pages.